# Решение: CTR группы за день на if'ах

Сырая строка в выгрузке — клетка отчёта, не объявление и не день.
Здесь объект другой: **группа объявлений за один день**.

Цель — доля кликов группы за день, не среднее `CTR (%)` по строкам:

$$
\mathrm{CTR} = 100 \times \frac{\sum \text{клики}}{\sum \text{показы}}
$$

Таблица `group_day` уже собрана. Напишите `predict_ctr(row)` из обычных `if`.


In [1]:
import pandas as pd
import numpy as np

from look_at_data_checks import (
    check_baseline,
    make_group_day,
    split_train_test,
    dummy_ctr,
)

raw = pd.read_csv("../dataset/bycicles.csv")
raw["Дата"] = pd.to_datetime(raw["Дата"], format="%d.%m.%Y")

group_day = make_group_day(raw)
train, test = split_train_test(group_day)
print("поезд", train.shape, "тест", test.shape)
print("константа CTR", round(dummy_ctr(train), 2))
group_day.head(3)


поезд (187, 6) тест (195, 6)
константа CTR 11.14


,Дата,Группа,показы,клики,CTR,месяц
0,2023-06-15,Велик ГЕО,5,1,20.000000,6
1,2023-06-15,Велик Купить / Цена,13,1,7.692308,6
2,2023-06-15,Веломагазин,36,5,13.888889,6


## Правила

- в зачёт идут дни с `показы ≥ 10` (иначе CTR снова 0 или 100);
- учимся на 2023, проверяемся на 2024;
- константа — средний CTR поезда по сумме кликов и показов;
- ваша функция должна дать **меньший MAE**, чем эта константа;
- ответ — число от 0 до 100.

`if` может смотреть `Группа` и `месяц`.

Средний CTR каждой из 25 групп на 2023 часто проигрывает константе: в 2024 детские кликают иначе.
Надёжнее грубые корзины (детские / города / остальное), а не 25 отдельных средних.


In [2]:
KIDS = {
    "Велосипед Детский",
    "Велосипед Девочке",
    "Велосипед Ребенку",
    "Велосипед Трехколесный",
}
GEO = {"Велосипед Ангарск", "Велосипед Братск"}
BRAND = {
    "Велосипед Stels",
    "Велосипед Stinger",
    "Велосипед Favorit",
    "Велосипед Novatrack",
}


def predict_ctr(row):
    group = row["Группа"]
    if group in KIDS:
        ctr = 18.0
    elif group in GEO:
        ctr = 16.0
    elif group == "Веломагазин":
        ctr = 12.0
    elif group in BRAND:
        ctr = 11.0
    elif group == "Велосипед Купить":
        ctr = 9.0
    else:
        ctr = 10.0

    if row["месяц"] == 4:
        ctr -= 1.0
    if row["месяц"] in (6, 7):
        ctr += 0.5
    return float(np.clip(ctr, 0, 100))


In [3]:
check_baseline(group_day, predict_ctr)


константа (CTR поезда) = 11.14, MAE = 6.159
ваши if'ы, MAE = 5.752
Бейзлайн: ок
